# Lecture 4 — Nonlinear Forecasting Methods
**ECON 417 · Business Forecasting · Southern Illinois University Edwardsville**

In Lectures 1–3 we built linear models and learned to regularise them.  
Today we ask: *what if the relationship between predictors and returns is not a straight line?*

Financial relationships are often nonlinear:  
- A 1-point VIX spike has a small effect when markets are calm, but a large effect during a crisis.  
- The beta of a tech stock to the market is higher in stressed periods than in calm ones.  
- Simple if-then rules ("if VIX > 30 and the market fell yesterday…") capture regime behaviour that linear models miss.

| Section | Topic |
|---------|-------|
| A | Curvature — VIX and returns, polynomial terms |
| B | Interaction terms — VIX regime × MSFT beta |
| C | Decision trees — regime-splitting if-then rules |
| D | Overfitting with trees — tree depth as regularization |
| E | Random forest — stability through averaging |
| F | Feature importance and RF as a risk-control tool |

## Setup

In [ ]:
%pip install yfinance scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Consistent colour palette (matches L1–L3) ────────────────────────────
COLORS = {
    'train':   '#2196F3',   # blue
    'val':     '#FF9800',   # orange
    'test':    '#4CAF50',   # green
    'ols':     '#E53935',   # red
    'ridge':   '#7B1FA2',   # purple
    'lasso':   '#00897B',   # teal
    'neutral': '#546E7A',   # slate
    'accent':  '#F4A261',   # warm orange
}

# ── Graph output directory ───────────────────────────────────────────────
GRAPH_DIR = '../Lecture 4/graph'
os.makedirs(GRAPH_DIR, exist_ok=True)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(np.asarray(a), np.asarray(b))))

In [ ]:
# ── Download financial data (same dataset as L1–L3) ──────────────────────
DATA_SOURCE = 'synthetic (calibrated to real S&P 500 properties)'
raw = None
try:
    import yfinance as yf
    tickers = ['^GSPC', '^VIX', 'MSFT', 'AAPL', 'JPM', 'XOM']
    _raw = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)
    if len(_raw) > 100:
        raw = _raw
        DATA_SOURCE = 'real (yfinance)'
except Exception:
    pass

if raw is not None:
    close = raw['Close']
    sp500 = close['^GSPC']
    vix   = raw['Close']['^VIX'].reindex(sp500.index).ffill()
    msft  = close['MSFT']
    aapl  = close['AAPL']
    jpm   = close['JPM']
    xom   = close['XOM']
else:
    np.random.seed(42)
    dates = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    N = len(dates)
    sp500_ret = np.random.normal(0.0003, 0.010, N)
    sp500 = pd.Series(4000 * np.exp(np.cumsum(sp500_ret)), index=dates)
    vix_vals = 18 + np.cumsum(np.random.normal(0, 0.2, N) - 0.3 * sp500_ret * 10)
    vix_vals = np.clip(vix_vals, 10, 80)
    vix = pd.Series(vix_vals, index=dates)
    def make_stock(base, beta, idio):
        r = beta * sp500_ret + np.random.normal(0, idio, N)
        return pd.Series(base * np.exp(np.cumsum(r)), index=dates)
    msft = make_stock(300, 1.2, 0.012)
    aapl = make_stock(150, 1.15, 0.013)
    jpm  = make_stock(80,  1.05, 0.011)
    xom  = make_stock(60,  0.80, 0.014)

df = pd.DataFrame({'SP500': sp500, 'VIX': vix,
                   'MSFT': msft, 'AAPL': aapl, 'JPM': jpm, 'XOM': xom}).dropna()

rets = np.log(df[['SP500','MSFT','AAPL','JPM','XOM']]).diff().dropna()
rets['VIX']     = df['VIX'].reindex(rets.index)
rets['VIX_chg'] = df['VIX'].diff().reindex(rets.index)
rets = rets.dropna()

print(f"Data source : {DATA_SOURCE}")
print(f"Date range  : {rets.index[0].date()} → {rets.index[-1].date()}")
print(f"Observations: {len(rets):,}")
rets.head(3)

In [ ]:
# ── Build lagged feature matrix (all predictors lagged 1 day) ────────────
# Rationale: we can only use information available BEFORE the return is realised.
# On day t, we predict SP500_ret(t) using features from day t-1.

X_all = pd.DataFrame({
    'lag_SP500': rets['SP500'].shift(1),
    'lag_MSFT':  rets['MSFT'].shift(1),
    'lag_AAPL':  rets['AAPL'].shift(1),
    'lag_JPM':   rets['JPM'].shift(1),
    'lag_XOM':   rets['XOM'].shift(1),
    'VIX':       rets['VIX'].shift(1),      # VIX level at prior close
    'VIX_chg':   rets['VIX_chg'].shift(1),  # VIX change on prior day
}).dropna()
y_all = rets['SP500'].reindex(X_all.index)

# 70 / 15 / 15 chronological split (same convention as L2, L3)
T    = len(X_all)
n_tr = int(T * 0.70)
n_va = int(T * 0.15)
n_te = T - n_tr - n_va

X_tr  = X_all.iloc[:n_tr];                y_tr  = y_all.iloc[:n_tr]
X_val = X_all.iloc[n_tr:n_tr+n_va];      y_val = y_all.iloc[n_tr:n_tr+n_va]
X_te  = X_all.iloc[n_tr+n_va:];          y_te  = y_all.iloc[n_tr+n_va:]
FEATURES = list(X_all.columns)

print(f"Train : {len(y_tr):4d} obs  ({y_tr.index[0].date()} → {y_tr.index[-1].date()})")
print(f"Val   : {len(y_val):4d} obs  ({y_val.index[0].date()} → {y_val.index[-1].date()})")
print(f"Test  : {len(y_te):4d} obs  ({y_te.index[0].date()} → {y_te.index[-1].date()})")
print(f"Features: {FEATURES}")

---
## Section A — Why Linear Models Can Miss Financial Reality

### The VIX is not a linear fear gauge

The VIX (CBOE Volatility Index) measures expected S&P 500 volatility.  
A linear model would say: *every 1-point VIX rise causes the same drop in returns.*  
But real data tell a different story:

- When VIX moves from 13 to 14 (calm period): market barely reacts.
- When VIX moves from 30 to 31 (crisis period): market drops sharply.

This is the **nonlinearity** the linear model misses.  
Adding a quadratic term — $\text{VIX\_chg}^2$ — lets the model capture this amplification.

The scatter plot below shows the *contemporaneous* relationship between daily VIX change and S&P 500 return.  
We use the same-day relationship here purely to illustrate the shape; in our predictive models we will always lag predictors by one day.

In [ ]:
# ── Example 4.1 — Polynomial: VIX change vs. S&P 500 return ─────────────
# Contemporaneous relationship shown for shape illustration only

vix_c   = rets['VIX_chg']
sp500_c = rets['SP500']

# Colour-code scatter by VIX regime
vix_lvl = rets['VIX']
calm     = vix_lvl < 17
normal   = (vix_lvl >= 17) & (vix_lvl < 28)
stressed = vix_lvl >= 28

# Fit linear and quadratic models
X_v  = vix_c.values.reshape(-1, 1)
pf   = PolynomialFeatures(degree=2, include_bias=False)
X_vp = pf.fit_transform(X_v)

lin_m  = LinearRegression().fit(X_v,  sp500_c)
quad_m = LinearRegression().fit(X_vp, sp500_c)

xg = np.linspace(vix_c.quantile(0.01), vix_c.quantile(0.99), 300).reshape(-1, 1)
yg_lin  = lin_m.predict(xg)
yg_quad = quad_m.predict(pf.transform(xg))

r2_lin  = lin_m.score(X_v,  sp500_c)
r2_quad = quad_m.score(X_vp, sp500_c)

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(vix_c[calm],     sp500_c[calm],     s=7, alpha=0.35, color=COLORS['test'],    label='Calm  (VIX < 17)')
ax.scatter(vix_c[normal],   sp500_c[normal],   s=7, alpha=0.35, color=COLORS['neutral'], label='Normal (VIX 17–28)')
ax.scatter(vix_c[stressed], sp500_c[stressed], s=7, alpha=0.45, color=COLORS['ols'],     label='Stressed (VIX > 28)')
ax.plot(xg, yg_lin,  color=COLORS['ols'],   lw=2.5, label=f'Linear fit  (R²={r2_lin:.3f})',    zorder=4)
ax.plot(xg, yg_quad, color=COLORS['train'], lw=2.5, ls='--', label=f'Quadratic fit (R²={r2_quad:.3f})', zorder=4)
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_xlabel('Daily VIX Change', fontsize=12)
ax.set_ylabel('S&P 500 Daily Log Return', fontsize=12)
ax.set_title('Example 4.1 — VIX Change vs. S&P 500 Return: Linear vs. Polynomial Fit\n'
             'Extreme VIX spikes (stressed regime) hit returns harder than a straight line predicts',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_1_polynomial_vs_linear.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Linear R² = {r2_lin:.4f}  |  Quadratic R² = {r2_quad:.4f}")
print("fig_4_1 saved.")

**What to notice:**  
The blue dashed (quadratic) line bends downward more sharply on the right — capturing the extra pain of large VIX spikes that the red linear line misses.  
The improvement in R² is modest on average, but the *shape* matters most for risk communication:  
a linear model systematically understates how bad a panic day can be.

**Forecasting implication:** Before dismissing nonlinear terms, ask whether the *tails* of your distribution matter to the decision.  
For a risk manager, they almost always do.

---
## Section B — Interaction Terms: VIX Regime × MSFT Beta

### Correlation convergence in a crisis

In calm markets, stocks move somewhat independently — MSFT can rise while the S&P 500 drifts.  
In stressed markets (high VIX), everything falls together.  
The correlation between individual stocks and the index *spikes during crises* — a well-documented phenomenon in finance.

A linear model with only `lag_MSFT` as a predictor gives one fixed slope for all market conditions.  
An interaction term `lag_MSFT × high_VIX` lets the slope *change* depending on the market regime:

$$\hat{y}_t = \beta_0 + \beta_1 \cdot \text{lag\_MSFT} + \beta_2 \cdot \text{high\_VIX} + \beta_3 \cdot (\text{lag\_MSFT} \times \text{high\_VIX})$$

- In calm conditions (high\_VIX = 0): slope of MSFT = $\beta_1$
- In stressed conditions (high\_VIX = 1): slope of MSFT = $\beta_1 + \beta_3$

We expect $\beta_3 > 0$: MSFT's predictive slope for S&P 500 is steeper during crises.

In [ ]:
# ── Example 4.2 — Interaction: VIX regime × MSFT lagged return ───────────

high_vix = (X_all['VIX'] >= 25).astype(int)  # VIX ≥ 25 = stressed regime

X_inter = pd.DataFrame({
    'lag_MSFT':  X_all['lag_MSFT'],
    'high_vix':  high_vix,
    'msft_hvix': X_all['lag_MSFT'] * high_vix,
})

# Fit main-effects-only model and interaction model
m_main  = LinearRegression().fit(X_inter[['lag_MSFT', 'high_vix']], y_all)
m_inter = LinearRegression().fit(X_inter, y_all)

b0, b1_main, b2_main = m_main.intercept_, m_main.coef_[0], m_main.coef_[1]
b0i, b1i, b2i, b3i  = m_inter.intercept_, *m_inter.coef_

print("Main-effects model:    SP500 ~ lag_MSFT + high_VIX")
print(f"  MSFT slope (all regimes): {b1_main:.5f}")
print()
print("Interaction model:     SP500 ~ lag_MSFT + high_VIX + lag_MSFT × high_VIX")
print(f"  MSFT slope in CALM market  (high_VIX=0): {b1i:.5f}")
print(f"  MSFT slope in STRESS market (high_VIX=1): {b1i + b3i:.5f}")
print(f"  Interaction coefficient β₃ = {b3i:.5f}  ("+(">0 ✓ steeper in crisis" if b3i > 0 else "<0")+ ")")

# ── Plot ─────────────────────────────────────────────────────────────────
msft_range = np.linspace(X_all['lag_MSFT'].quantile(0.03),
                          X_all['lag_MSFT'].quantile(0.97), 200)

fig, ax = plt.subplots(figsize=(12, 6))

# Scatter coloured by regime
mask_lo = high_vix == 0
mask_hi = high_vix == 1
ax.scatter(X_all.loc[mask_lo, 'lag_MSFT'], y_all.loc[mask_lo],
           s=7, alpha=0.20, color=COLORS['test'],    label='_nolegend_')
ax.scatter(X_all.loc[mask_hi, 'lag_MSFT'], y_all.loc[mask_hi],
           s=7, alpha=0.25, color=COLORS['ols'],     label='_nolegend_')

# Predicted lines for each regime
for hvix_val, label, color, ls in [
    (0, 'Calm market (VIX < 25)',     COLORS['test'], '-'),
    (1, 'Stressed market (VIX ≥ 25)', COLORS['ols'],  '--'),
]:
    Xp = pd.DataFrame({'lag_MSFT': msft_range, 'high_vix': hvix_val,
                        'msft_hvix': msft_range * hvix_val})
    ax.plot(msft_range, m_inter.predict(Xp), lw=2.8, color=color, ls=ls, label=label, zorder=4)

ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
ax.axvline(0, color='black', lw=0.7, ls='--', alpha=0.4)
ax.set_xlabel('MSFT Lagged Return (day t−1)', fontsize=12)
ax.set_ylabel('S&P 500 Return (day t)', fontsize=12)
ax.set_title('Example 4.2 — Interaction Effect: MSFT Beta Changes With VIX Regime\n'
             'Steeper slope in stressed markets = correlation convergence in a crisis',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
pct_stressed = mask_hi.mean() * 100
ax.annotate(f'{pct_stressed:.0f}% of days in stressed regime',
            xy=(0.02, 0.05), xycoords='axes fraction', fontsize=9, color=COLORS['ols'])
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_2_interaction_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_4_2 saved.")

**Key observation:**  
The two lines have different slopes — MSFT's predictive relationship with the S&P 500 is steeper when VIX is elevated.  
A model without the interaction term forces one slope for all market conditions, underestimating tail co-movement.

**Business interpretation:**  
A risk manager building a portfolio stress test should use the crisis-regime slope, not the average slope.  
The interaction term makes this distinction explicit in the model.

---
## Section C — Decision Trees: Forecasting With If-Then Rules

### Regimes without manual thresholds

A decision tree automates the regime-detection task:  
instead of us choosing "VIX > 25" as a threshold, the tree *learns* the most informative split from data.

A depth-3 tree on two inputs produces up to 8 rectangular **partitions** of the feature space,  
each with its own predicted return.  
The regions tell a story:

> "If VIX spiked yesterday AND the market fell yesterday → predict a small negative return."  
> "If VIX fell AND the market rose → predict a small positive return."

This is interpretable — and is the core appeal of trees for financial practitioners.

In [ ]:
# ── Example 4.3 — Decision tree partitions on (lag_SP500, VIX_chg) ───────

tree2d = DecisionTreeRegressor(max_depth=3, random_state=417)
tree2d.fit(X_all[['lag_SP500', 'VIX_chg']], y_all)

# Meshgrid for visualisation
sp_range  = np.linspace(X_all['lag_SP500'].quantile(0.01), X_all['lag_SP500'].quantile(0.99), 180)
vc_range  = np.linspace(X_all['VIX_chg'].quantile(0.01),  X_all['VIX_chg'].quantile(0.99),  180)
XX, YY = np.meshgrid(sp_range, vc_range)
ZZ = tree2d.predict(np.column_stack([XX.ravel(), YY.ravel()])).reshape(XX.shape)

vmax = np.percentile(np.abs(ZZ), 98)

fig, ax = plt.subplots(figsize=(11, 7))
cf = ax.contourf(XX, YY, ZZ, levels=20, cmap='RdYlGn', vmin=-vmax, vmax=vmax, alpha=0.80)
cb = plt.colorbar(cf, ax=ax)
cb.set_label('Predicted S&P 500 Return', fontsize=10)

# Training data overlay
ax.scatter(X_tr['lag_SP500'], X_tr['VIX_chg'],
           c=y_tr.values, cmap='RdYlGn', vmin=-vmax, vmax=vmax,
           s=10, alpha=0.55, edgecolors='white', linewidths=0.3, zorder=3)

ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.5, label='VIX unchanged')
ax.axvline(0, color='black', lw=1.0, ls='--', alpha=0.5, label='Market flat')

# Annotate quadrants
for x_ann, y_ann, txt, col in [
    (-0.025,  4, 'Market↓ VIX↑\n→ predict negative', COLORS['ols']),
    ( 0.020,  4, 'Market↑ VIX↑\n→ ambiguous',         'gray'),
    (-0.025, -3, 'Market↓ VIX↓\n→ ambiguous',          'gray'),
    ( 0.020, -3, 'Market↑ VIX↓\n→ predict positive',   COLORS['test']),
]:
    ax.text(x_ann, y_ann, txt, fontsize=8.5, color=col,
            ha='center', va='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

ax.set_xlabel('Lagged S&P 500 Return (day t−1)', fontsize=12)
ax.set_ylabel('Lagged VIX Change (day t−1)', fontsize=12)
ax.set_title('Example 4.3 — Decision Tree Partitions: Regime-Specific Forecast Rules\n'
             'Colours show predicted return; tree depth = 3 creates up to 8 rectangular regions',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_3_tree_partitions.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_4_3 saved.")

**Key observation:**  
The tree has automatically found that the upper-left corner (market fell, VIX rose) is the most bearish regime,  
and the lower-right corner (market rose, VIX fell) is the most bullish.  
The boundaries are axis-aligned (horizontal and vertical lines), which is the defining feature of decision trees.  

**Limitation:** The rectangular partitions can look blocky and may miss diagonal decision boundaries.  
Random forest (Section E) softens this by averaging many trees with different splits.

---
## Section D — Tree Depth, Flexibility, and Overfitting

### More depth = more flexibility = more overfitting risk

A depth-1 tree (one split, two leaves) is barely flexible — it makes a binary prediction.  
A depth-10 tree can have up to 1,024 leaves.  
With ~1,760 training days, a depth-10 tree fits roughly 1–2 observations per leaf — memorising noise.

**Tree depth is a form of regularization** — the same logic as λ in Ridge and Lasso from Lecture 3:

| Tool | Controls | Effect |
|------|----------|--------|
| λ in Ridge/Lasso | Coefficient magnitude | Reduces variance |
| `max_depth` in tree | Number of splits | Reduces variance |
| `min_samples_leaf` | Minimum leaf size | Reduces variance |

Hyperparameter selection always uses validation performance, never in-sample fit.

In [ ]:
# ── Example 4.4 — Tree depth vs. validation RMSE ─────────────────────────

depths = list(range(1, 14))
tr_rmse_list  = []
val_rmse_list = []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=417)
    m.fit(X_tr, y_tr)
    tr_rmse_list.append(rmse(y_tr,  m.predict(X_tr)))
    val_rmse_list.append(rmse(y_val, m.predict(X_val)))

best_depth = depths[int(np.argmin(val_rmse_list))]

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(depths, tr_rmse_list,  marker='o', color=COLORS['train'], lw=2.2, label='Train RMSE')
ax.plot(depths, val_rmse_list, marker='o', color=COLORS['val'],   lw=2.2, label='Validation RMSE')
ax.axvline(best_depth, color=COLORS['test'], lw=2, ls='--',
           label=f'Best validation depth = {best_depth}')
ax.fill_between([best_depth - 0.4, max(depths) + 0.4],
                 min(tr_rmse_list) - 0.0001, max(val_rmse_list) + 0.0001,
                 alpha=0.07, color=COLORS['ols'], label='Overfitting zone')
ax.set_xlabel('Tree Depth (max_depth)', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Example 4.4 — Tree Depth vs. Forecast Error\n'
             'Train RMSE falls monotonically; validation RMSE bottoms out then rises (overfitting)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(depths)
ax.annotate(f'Optimal: depth={best_depth}',
            xy=(best_depth, min(val_rmse_list)),
            xytext=(best_depth + 1.5, min(val_rmse_list) + 0.0002),
            fontsize=10, color=COLORS['test'],
            arrowprops=dict(arrowstyle='->', color=COLORS['test']))
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_4_depth_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best depth by validation = {best_depth}")
print(f"Val RMSE at best depth   = {min(val_rmse_list):.6f}")

**Key takeaway:**  
Lecture 2's central lesson — *train RMSE always improves with complexity; validation RMSE tells you the real story* —  
applies identically to decision trees.  
The optimal depth selected by validation is the depth we carry forward into random forest hyperparameter settings.

---
## Section E — Random Forest: Stability Through Averaging

### The instability problem with a single tree

A single decision tree is volatile: a small change in the training data can produce a very different set of splits and a different forecast.  
This is the same *variance* problem we saw with OLS under multicollinearity — and the fix is analogous.

**Random forest** reduces variance by averaging predictions from many trees:  
1. Draw a random bootstrap sample of the training data.  
2. At each split, consider only a random subset of features (typically $\sqrt{p}$ features).  
3. Grow a full tree on that subset.  
4. Repeat 200–500 times, then average all predictions.

The averaging smooths out the idiosyncratic noise of each individual tree.  
Result: lower variance, more stable forecasts — better suited for institutional use.

| | Single decision tree | Random forest |
|--|-----|------|
| Interpretability | High (readable splits) | Lower (average of many) |
| Variance | High (sensitive to data) | Lower (averaging stabilises) |
| Out-of-sample RMSE | Often weaker | Usually stronger |
| Institutional use | Limited | Preferred for risk-sensitive decisions |

In [ ]:
# ── Example 4.5 — Single tree vs. random forest ───────────────────────────

best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=417)
best_tree.fit(X_tr, y_tr)

forest = RandomForestRegressor(n_estimators=200, max_depth=best_depth + 2,
                                max_features='sqrt', random_state=417)
forest.fit(X_tr, y_tr)

pred_tree   = best_tree.predict(X_te)
pred_forest = forest.predict(X_te)

rmse_tree   = rmse(y_te, pred_tree)
rmse_forest = rmse(y_te, pred_forest)

# ── Plot test-period forecast path (show first 80 days for clarity) ───────
W = min(80, len(y_te))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [3, 1]})

ax = axes[0]
ax.plot(y_te.index[:W], y_te.values[:W],
        color=COLORS['neutral'], lw=1.5, label='Actual SP500 return', alpha=0.9)
ax.plot(y_te.index[:W], pred_tree[:W],
        color=COLORS['val'], lw=1.8, label=f'Single tree (depth {best_depth})', alpha=0.85)
ax.plot(y_te.index[:W], pred_forest[:W],
        color=COLORS['train'], lw=1.8, label='Random forest (200 trees)', alpha=0.85)
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('S&P 500 Log Return', fontsize=11)
ax.set_title('Test-Period Forecast: Single Tree vs. Random Forest\n'
             'Forest forecast is smoother — individual tree noise averaged away',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
bars = ax.bar(['Single\nTree', 'Random\nForest'], [rmse_tree, rmse_forest],
               color=[COLORS['val'], COLORS['train']], edgecolor='white', width=0.5)
ax.set_ylabel('Test RMSE', fontsize=11)
ax.set_title('Test RMSE\nComparison', fontsize=12, fontweight='bold')
for bar, val in zip(bars, [rmse_tree, rmse_forest]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.003,
            f'{val:.5f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(rmse_tree, rmse_forest) * 1.2)

fig.suptitle('Example 4.5 — Single Tree vs. Random Forest: S&P 500 Return Forecasting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_5_tree_vs_forest.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Single tree test RMSE  = {rmse_tree:.6f}")
print(f"Random forest test RMSE = {rmse_forest:.6f}")
print(f"RMSE improvement       = {(rmse_tree - rmse_forest)/rmse_tree*100:.1f}%")

---
## Section F — Feature Importance and RF as a Risk-Control Tool

### Which inputs matter most?

Random forests automatically rank predictors by their contribution to reducing forecast error across all trees.  
In `sklearn`, `feature_importances_` gives the average decrease in node impurity (variance) explained by each feature, normalised to sum to 1.

**Caution:** Two highly correlated predictors split importance between them (similar to the multicollinearity issue in L3).  
Importance scores tell you what the *model uses*, not what *causes* returns.

### Why institutions prefer random forests

Institutions — banks, asset managers, risk committees — are risk-averse about their *models*, not just their portfolios.  
A single decision tree can change its forecast dramatically when one unusual day is added to the training window.  
Random forest averages that noise away: the forecast is more stable period-to-period, which makes it easier to defend in risk committees and regulatory reviews.

In [ ]:
# ── Example 4.6 — Feature importance in random forest ────────────────────

# Refit with all features
rf_full = RandomForestRegressor(n_estimators=300, max_depth=best_depth + 2,
                                 max_features='sqrt', random_state=417)
rf_full.fit(X_tr, y_tr)

imp_df = (pd.DataFrame({'Feature': FEATURES, 'Importance': rf_full.feature_importances_})
            .sort_values('Importance', ascending=True))

# Colour by predictor type
def feat_color(f):
    if 'VIX' in f: return COLORS['ols']
    if 'SP500' in f: return COLORS['train']
    return COLORS['neutral']

bar_colors = [feat_color(f) for f in imp_df['Feature']]

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
                color=bar_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, imp_df['Importance']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2.,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('Feature Importance (mean node impurity decrease, normalised)', fontsize=11)
ax.set_title('Example 4.6 — Random Forest Feature Importance\n'
             'Which predictors drive S&P 500 return forecasts?',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, imp_df['Importance'].max() * 1.25)

legend_handles = [
    mpatches.Patch(facecolor=COLORS['ols'],    label='Volatility predictors (VIX level, VIX change)'),
    mpatches.Patch(facecolor=COLORS['train'],  label='Own momentum (lagged SP500 return)'),
    mpatches.Patch(facecolor=COLORS['neutral'],label='Sector stock returns (MSFT, AAPL, JPM, XOM)'),
]
ax.legend(handles=legend_handles, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Feature importances (high to low):")
print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))

In [ ]:
# ── RF stability: rolling window variance vs. single tree ─────────────────
# Shows that RF produces more consistent predictions over time

WINDOW = 120
tree_rolling_std   = []
forest_rolling_std = []
roll_dates         = []

for end in range(WINDOW, len(X_all) - 20, 20):   # step by 20 days for speed
    Xw = X_all.iloc[end - WINDOW : end]
    yw = y_all.iloc[end - WINDOW : end]
    Xf = X_all.iloc[end : end + 20]

    t_m = DecisionTreeRegressor(max_depth=best_depth, random_state=417).fit(Xw, yw)
    f_m = RandomForestRegressor(n_estimators=100, max_depth=best_depth+2,
                                 max_features='sqrt', random_state=417).fit(Xw, yw)

    tree_rolling_std.append(np.std(t_m.predict(Xf)))
    forest_rolling_std.append(np.std(f_m.predict(Xf)))
    roll_dates.append(X_all.index[end])

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(roll_dates, tree_rolling_std,   color=COLORS['val'],   lw=1.8, label='Single tree (std of predictions)', alpha=0.85)
ax.plot(roll_dates, forest_rolling_std, color=COLORS['train'], lw=1.8, label='Random forest (std of predictions)', alpha=0.85)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Std dev of 20-day forecast', fontsize=11)
ax.set_title('Random Forest as a Risk-Control Tool\n'
             'Forest forecasts vary less over time — key for institutional model governance',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_4_7_rf_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_4_7 saved.")
print(f"Mean rolling forecast std — Single tree: {np.mean(tree_rolling_std):.5f}")
print(f"Mean rolling forecast std — Random forest: {np.mean(forest_rolling_std):.5f}")

---
## Forecasting Rules for Nonlinear Methods

The lesson from this lecture is **not** "trees beat linear models" or "random forest is always better."  
The correct lesson has three parts:

1. **Start with the simplest model that beats baselines out of sample.**  
   A ridge regression from L3 already captures most of the linear signal in financial returns.

2. **Add complexity only when validation results clearly justify it.**  
   The tree depth curve (Section D) is your guide — not in-sample R².

3. **Use random forest when the dataset is large, simpler models leave clear unexplained structure, and the deployment context tolerates lower interpretability.**  
   For daily returns (nearly unpredictable), the improvement over ridge may be small.  
   For longer-horizon signals with rich feature sets, the improvement can be substantial.

---
## Interview Preparation

**"Why did you choose random forest?"**  
> "Random forest reduces the instability of a single decision tree by averaging many trees fitted on different bootstrap samples and random feature subsets.  
> It handles nonlinear effects and interactions automatically — for example, VIX's effect on returns changing in high-stress regimes — without requiring manual feature engineering.  
> It is also more stable than any single tree: less sensitive to one unusual trading day, which matters when the model is reviewed by a risk committee."

**"What does random forest buy you over linear models?"**  
> "Linear models assume the effect of a predictor stays constant across its full range.  
> Random forest allows the effect to vary depending on the values of other variables — for example, MSFT's predictive slope for the S&P 500 is steeper when VIX is elevated (correlation convergence in a crisis).  
> This is more realistic for financial relationships that have regimes and thresholds."

**"What are the limitations of random forest?"**  
> "Three main limitations.  
> First, it is harder to explain than a single tree or a linear model — there is no one-sentence summary of the model logic.  
> Second, it does not extrapolate well beyond the training data range: if the market enters a regime it has never seen (VIX > 80, as in early COVID), the forest can only predict within the distribution it has trained on.  
> Third, like all models, it must be validated out of sample and monitored after deployment; strong validation performance does not guarantee future stability."